# Malignancy ratings and patient-level splits

## Goal

Understand the reader malignancy ratings in LIDC and LNDb, compare mean, median and mode, and save the targets and patient-level folds. We keep both an all-size LNDb dataset for the competition and a ≥3 mm version for comparison with LIDC.

## Load the data

For LIDC, `lidc_annotations_clean.csv` contains one row per reader annotation after removing the 14 ambiguous clusters.

For LNDb, we need `allNods.csv` and `chars_trainNodules.csv`, because the first one tells us which annotations refer to the same nodule and whether each radiologist considered it a nodule, while the second one contains each radiologist’s individual malignancy rating. This allows us to group the ratings by nodule and calculate our own mean, median, and mode.

In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
#CHANGE THIS TO THE LOCATION ON YOUR MACHINE:
LNDB_DIR = Path(os.environ.get("LNDB_DIR", Path.home() / "Downloads" / "LNDb"))
sys.path.insert(0, str(ROOT)) #add the root to the path so python can find the things inside src/

from src.malignancy import prepare_lndb_annotations, aggregate_ratings, create_targets
from src.splitting import create_patient_folds

`allNods.csv` combines findings from two sources: annotations made directly on the CT images and information extracted from medical reports. We will only use the image annotations because `chars_trainNodules.csv` provides individual radiologist malignancy ratings only for those findings. Report-only findings have no RadID and are removed.

When radiologists disagree, we keep ratings from those who considered the finding a nodule. Nodule=0 provides no valid malignancy rating and is not included in the aggregation.

In [2]:
lidc = pd.read_csv(ROOT / "data/processed/lidc_annotations_clean.csv")
lndb_chars = pd.read_csv(LNDB_DIR / "chars_trainNodules.csv")
lndb_all = pd.read_csv(LNDB_DIR / "allNods.csv")
lndb_all_sizes = prepare_lndb_annotations(lndb_chars, lndb_all)
assert lndb_all_sizes["malignancy"].between(1, 5).all()

In [3]:
display(lidc.loc[
    lidc["diameter"] < 3,
    ["patient_id", "series_uid", "nodule_id", "diameter"],
])

lndb_sizes = lndb_all_sizes.drop_duplicates(["LNDbID", "NoduleID"])

print("LNDb <3 mm:", lndb_sizes["DiamEq_Rad"].eq(3).sum())
print("LNDb segmented:", lndb_sizes["DiamEq_Rad"].gt(3).sum())

,patient_id,series_uid,nodule_id,diameter
5057,LIDC-IDRI-0899,1.3.6.1.4.1.14519.5.2.1.6279.6001.216652640878...,0,2.878157
5350,LIDC-IDRI-0857,1.3.6.1.4.1.14519.5.2.1.6279.6001.401389720232...,1,2.795085
5711,LIDC-IDRI-0801,1.3.6.1.4.1.14519.5.2.1.6279.6001.768276876111...,2,2.797761
5793,LIDC-IDRI-0786,1.3.6.1.4.1.14519.5.2.1.6279.6001.259453428008...,0,2.544731
6004,LIDC-IDRI-0759,1.3.6.1.4.1.14519.5.2.1.6279.6001.692598144815...,3,2.585936


LNDb <3 mm: 143
LNDb segmented: 622


In [4]:
lndb = lndb_all_sizes.copy()
lndb_ge3 = lndb_all_sizes[lndb_all_sizes["DiamEq_Rad"] > 3].copy()
lndb_under_3 = lndb_all_sizes[lndb_all_sizes["DiamEq_Rad"] == 3].copy()

## Keep both LNDb datasets

We keep all 765 LNDb nodules for the competition. We also create a second dataset containing 622 nodules for comparison with LIDC, whose malignancy ratings come from the ≥3 mm annotation category. Some calculated LIDC diameters are slightly smaller, as shown above.

`DiamEq_Rad` is the equivalent diameter from the radiologist segmentation. In this table, `DiamEq_Rad = 3` marks the 143 point-only nodules smaller than 3 mm. We exclude them only from the second dataset so that this LNDb version matches LIDC's annotation size category. Their target distribution is shown separately below.

## Aggregate reader ratings

For each nodule, we calculate the mean, median and mode of the reader malignancy ratings and compare them as possible targets.

In [5]:
lidc_nodules = aggregate_ratings(lidc,["patient_id", "series_uid", "nodule_id"])
lndb_nodules = aggregate_ratings(lndb,["LNDbID", "NoduleID"])

In [6]:
lidc_nodules.head()

,patient_id,series_uid,nodule_id,ratings,malignancy_mean,malignancy_median,malignancy_mode
0,LIDC-IDRI-0001,1.3.6.1.4.1.14519.5.2.1.6279.6001.179049373636...,0,"[5, 5, 5, 4]",4.75,5.0,5.0
1,LIDC-IDRI-0002,1.3.6.1.4.1.14519.5.2.1.6279.6001.619372068417...,0,"[5, 4]",4.50,4.5,NaN
2,LIDC-IDRI-0003,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,0,[2],2.00,2.0,2.0
3,LIDC-IDRI-0003,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,1,"[5, 5, 3, 4]",4.25,4.5,5.0
4,LIDC-IDRI-0003,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,2,"[4, 4, 3, 2]",3.25,3.5,4.0


In [7]:
lndb_nodules[10:20]

,LNDbID,NoduleID,ratings,malignancy_mean,malignancy_median,malignancy_mode
10,3,1,"[2.0, 2.0]",2.000000,2.0,2.0
11,3,2,"[2.0, 1.0]",1.500000,1.5,NaN
12,3,3,"[2.0, 1.0]",1.500000,1.5,NaN
13,3,4,[1.0],1.000000,1.0,1.0
14,3,5,[1.0],1.000000,1.0,1.0
15,4,1,"[2.0, 2.0, 1.0]",1.666667,2.0,2.0
16,4,2,[1.0],1.000000,1.0,1.0
17,4,3,"[3.0, 1.0]",2.000000,2.0,NaN
18,5,1,[1.0],1.000000,1.0,1.0
19,5,2,[1.0],1.000000,1.0,1.0


## Compare mean, median and mode

Each nodule can have malignancy ratings from multiple radiologists, but each model requires one target per nodule. We therefore calculate the mean, median and mode and compare how they combine the reader ratings.

In [8]:
def aggregation_summary(dataset, nodules):
    rows = []
    for name in ["mean", "median", "mode"]:
        values = nodules[f"malignancy_{name}"].dropna()
        rows.append({
            "dataset": dataset,
            "aggregation": name,
            "available": len(values),
            "non-integer": int((~np.isclose(values, values.round())).sum()),
            "< 3": int(values.lt(3).sum()),
            "= 3": int(values.eq(3).sum()),
            "> 3": int(values.gt(3).sum()),
            "ties": len(nodules) - len(values) 
        })
    return rows

comparison = pd.DataFrame(
    aggregation_summary("LIDC", lidc_nodules)
    + aggregation_summary("LNDb", lndb_nodules)
)
comparison

,dataset,aggregation,available,non-integer,< 3,= 3,> 3,ties
0,LIDC,mean,2637,1146,1363,641,633,0
1,LIDC,median,2637,548,1149,952,536,0
2,LIDC,mode,2085,0,801,919,365,552
3,LNDb,mean,765,101,512,161,92,0
4,LNDb,median,765,78,505,171,89,0
5,LNDb,mode,653,0,430,155,68,112


The mode will not be used because ties leave many nodules without a unique target.

For binary classification, nodules with an aggregated malignancy value of 3 are commonly removed because they are considered indeterminate. Section 2 of [Baltatzis et al. (2021)](https://arxiv.org/abs/2108.05386) describes this as a common LIDC data-curation step.

Our results show the same pattern reported in the paper: the median produces more indeterminate nodules than the mean. Removing these nodules creates a smaller dataset that is easier to classify, particularly using nodule diameter.


## Create the targets

For the reasons presented above, we will create targets for three different prediction tasks:

- `target_regression_mean`: continuous mean.
- `target_regression_median`: continuous median.
- `target_classification_mean`: mean rounded to an integer from 1 to 5. Class 3 is retained.
- `target_binary_mean`: unrounded mean below 3 is benign and above 3 is malignant. Nodules with a mean exactly equal to 3 are excluded.

The regression targets retain decimal values created when radiologists give different ratings.

The integer target converts the mean back to the original 1–5 scale and keeps class 3. 

The binary target separates benign and malignant nodules using the unrounded mean and removes only nodules whose mean is exactly 3.

In [9]:
lidc_targets = create_targets(lidc_nodules)
lndb_targets = create_targets(lndb_nodules)
lndb_ge3_targets = create_targets(
    aggregate_ratings(lndb_ge3, ["LNDbID", "NoduleID"])
)

target_columns = [
    "target_regression_mean",
    "target_regression_median",
    "target_classification_mean",
    "target_binary_mean",
]

lidc_targets[["patient_id", "series_uid", "nodule_id"] + target_columns].head()

,patient_id,series_uid,nodule_id,target_regression_mean,target_regression_median,target_classification_mean,target_binary_mean
0,LIDC-IDRI-0001,1.3.6.1.4.1.14519.5.2.1.6279.6001.179049373636...,0,4.75,5.0,5,1
1,LIDC-IDRI-0002,1.3.6.1.4.1.14519.5.2.1.6279.6001.619372068417...,0,4.50,4.5,5,1
2,LIDC-IDRI-0003,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,0,2.00,2.0,2,0
3,LIDC-IDRI-0003,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,1,4.25,4.5,4,1
4,LIDC-IDRI-0003,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,2,3.25,3.5,3,1


In [10]:
lndb_under_3_targets = create_targets(
    aggregate_ratings(lndb_under_3, ["LNDbID", "NoduleID"])
)

## Target distributions

The regression and integer classification targets retain every nodule in each dataset. The binary target excludes nodules whose mean malignancy is exactly 3. We compare the class counts for LIDC, both LNDb versions, and the 143 smaller LNDb nodules to show what changes when they are excluded.

In [11]:
rows = []

for dataset, targets in [
    ("LIDC", lidc_targets),
    ("LNDb all sizes", lndb_targets),
    ("LNDb ≥3 mm", lndb_ge3_targets),
    ("LNDb <3 mm", lndb_under_3_targets),
]:
    integer_target = targets["target_classification_mean"]
    binary_target = targets["target_binary_mean"]

    rows.append({
        "dataset": dataset,
        "nodules": len(targets),
        "class 1": int(integer_target.eq(1).sum()),
        "class 2": int(integer_target.eq(2).sum()),
        "class 3": int(integer_target.eq(3).sum()),
        "class 4": int(integer_target.eq(4).sum()),
        "class 5": int(integer_target.eq(5).sum()),
        "binary benign": int(binary_target.eq(0).sum()),
        "binary malignant": int(binary_target.eq(1).sum()),
        "binary excluded": int(binary_target.isna().sum()),
    })

target_distribution = pd.DataFrame(rows)
target_distribution

,dataset,nodules,class 1,class 2,class 3,class 4,class 5,binary benign,binary malignant,binary excluded
0,LIDC,2637,286,608,1249,384,110,1363,633,641
1,LNDb all sizes,765,213,254,209,71,18,512,92,161
2,LNDb ≥3 mm,622,138,197,199,70,18,380,91,151
3,LNDb <3 mm,143,75,57,10,1,0,132,1,10


Among the 143 nodules <3 mm, 132 have benign binary targets, one malignant, and 10 are excluded as indeterminate. This imbalance could encourage the model to associate small size with benign ratings.

## Save the derived tables

Each CSV contains one row per nodule, with its identifiers, original reader ratings and selected targets. `lndb_nodule_targets.csv` contains all 765 LNDb nodules for the competition. `lndb_ge3_nodule_targets.csv` contains the 622 nodules for comparison with LIDC.

In [12]:
output_dir = ROOT / "data/processed"
lidc_targets[["patient_id", "series_uid", "nodule_id", "ratings", *target_columns]].to_csv(output_dir / "lidc_nodule_targets.csv", index=False)
lndb_targets[["LNDbID", "NoduleID", "ratings", *target_columns]].to_csv(output_dir / "lndb_nodule_targets.csv", index=False)
lndb_ge3_targets[["LNDbID", "NoduleID", "ratings", *target_columns]].to_csv(output_dir / "lndb_ge3_nodule_targets.csv", index=False)

## Create patient-level folds

The smallest integer class limits how many stratified folds can be created. LNDb has only 18 class-5 nodules, so we use five folds. LIDC has 110 class-5 nodules and a much larger dataset, so we use ten folds.

Because the split is performed by patient rather than by nodule, exact class balance is not guaranteed and is checked below. `StratifiedGroupKFold` keeps every patient in one fold while approximately preserving `target_classification_mean` ([scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)). The same fold assignments are reused for every target and model.

We create LNDb folds once using all 765 nodules. Both LNDb datasets use this patient-fold table, so removing smaller nodules does not move the remaining patients between folds.

In [13]:
lndb_folds = create_patient_folds(
    lndb_targets,
    "LNDbID",
    "target_classification_mean",
    n_splits=5,
)

lidc_folds = create_patient_folds(
    lidc_targets,
    "patient_id",
    "target_classification_mean",
    n_splits=10,
)

## Validate the folds

The checks below confirm that every patient has one fold and compare patient and class counts for LIDC and both LNDb versions. The class counts in each row also give the number of nodules in that fold.

In [14]:
lndb_folded = lndb_targets.merge(lndb_folds, on="LNDbID")

print("LNDb all sizes")
display(lndb_folds["fold"].value_counts().sort_index().rename("patients"))
pd.crosstab(
    lndb_folded["fold"],
    lndb_folded["target_classification_mean"],
)

LNDb all sizes


fold
1    44
2    48
3    39
4    42
5    39
Name: patients, dtype: int64

target_classification_mean,1,2,3,4,5
fold,,,,,
1,41,57,40,18,2
2,47,48,44,18,2
3,40,47,34,7,6
4,47,47,48,17,2
5,38,55,43,11,6


In [15]:
lndb_ge3_folded = lndb_ge3_targets.merge(lndb_folds, on="LNDbID", how="left")

print("LNDb ≥3 mm")
display(lndb_ge3_folded.groupby("fold")["LNDbID"].nunique().rename("patients"))
pd.crosstab(
    lndb_ge3_folded["fold"],
    lndb_ge3_folded["target_classification_mean"],
)

LNDb ≥3 mm


fold
1    41
2    47
3    37
4    42
5    37
Name: patients, dtype: int64

target_classification_mean,1,2,3,4,5
fold,,,,,
1,30,47,37,18,2
2,28,37,43,18,2
3,27,37,33,7,6
4,29,36,45,16,2
5,24,40,41,11,6


In [16]:
lidc_folded = lidc_targets.merge(lidc_folds, on="patient_id")

print("LIDC")
display(lidc_folds["fold"].value_counts().sort_index().rename("patients"))
pd.crosstab(
    lidc_folded["fold"],
    lidc_folded["target_classification_mean"],
)

LIDC


fold
1     86
2     84
3     87
4     90
5     87
6     87
7     84
8     88
9     88
10    89
Name: patients, dtype: int64

target_classification_mean,1,2,3,4,5
fold,,,,,
1,18,56,118,37,11
2,37,37,115,50,7
3,18,63,126,47,15
4,33,70,138,34,17
5,31,61,110,32,11
6,22,66,115,40,9
7,17,56,116,40,6
8,40,72,132,46,9
9,39,56,130,31,15


In [17]:
assert lndb_folds["LNDbID"].is_unique
assert lidc_folds["patient_id"].is_unique
assert lndb_ge3_folded["fold"].notna().all()
assert set(lndb_folds["fold"]) == set(range(1, 6))
assert set(lidc_folds["fold"]) == set(range(1, 11))

Class counts can differ because all nodules from the same patient must remain together. The ≥3 mm version can also have fewer patients in each fold, since patients with only smaller nodules are removed. The remaining patients keep their original fold numbers.

## Save the fold assignments

Each CSV contains only the patient or case identifier and its fold number. Both LNDb target CSVs use `lndb_patient_folds.csv`, joined on `LNDbID`. LIDC uses `lidc_patient_folds.csv`, joined on `patient_id`.

In [18]:
lndb_folds.to_csv(output_dir / "lndb_patient_folds.csv", index=False)
lidc_folds.to_csv(output_dir / "lidc_patient_folds.csv", index=False)